In [10]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

# ==== Paths taken from analysis2 / reviewrootnode / reviewbbpredict / feature notebooks ====
ROOT = Path(r"C:\Users\suuser\Desktop\minlplib")
FINAL = ROOT / 'Areview' / 'combined_minlplib_qplib'
SYNTHETIC = (ROOT / 'paperexperiments' / 'synthetic_prime_minimal_revision' /
             'bb_separate_trees' / 'synthetic_prime_8016_fixed100_v1' / 'postrun_clean')
# Final static ML run explicitly referenced in BOTH reviewrootnode and reviewbbgnn:
BB_RUN_NAME = 'sklearn_all_runs_synthetic_prime_8016_20261021_014902_034048'
BB_RUN = FINAL / BB_RUN_NAME
# Root runner timestamps each execution, so select the final exact directory here if multiple.
ROOT_RUN = None   # or FINAL / 'sklearn_root_synprime_8016_YYYYMMDD_...'
BB_GNN_RUN = FINAL / 'gnn_synprime_8016_20261021_020453_440418'  # user-confirmed B&B GNN run
ROOT_GNN_RUN = FINAL / 'gnn_ROOT_8_designs_UNBALANCED_20261024_165255_822932'  # verified from completed root GNN log

COMBINED_FEATURE_TIMES = FINAL / 'combined_minlplib_qplib_feature_times_v2.csv'
SYNTHETIC_FEATURE_TIMES = (ROOT / 'paperexperiments' / 'synthetic_prime_minimal_revision' /
                           'instances_mixed_8016_DI_feature_times.csv')
SUMMARY = FINAL / 'summary_combined_minlplib_qplib_BB_clean.csv'
TARGET = (FINAL / 'targets' / 'method2' / 'combined_minlplib_qplib' /
          'targets_method2_relaxation_combined_minlplib_qplib_probe10.csv')
# Actual LIVE file is produced by the combined post-run cleaner.
LIVE_CANDIDATES = [FINAL / 'combined_minlplib_qplib_BB_LIVE_clean.parquet',
                   FINAL / 'combined_minlplib_qplib_BB_LIVE_clean.csv']
OUT = FINAL / 'analysis3_gap_time_outputs_WITH_GNN'
ETA = 1e-3
TAU = 0.01

# ==== Strict utilities ====
def load(p):
    p=Path(p)
    if not p.exists(): raise FileNotFoundError(f'Missing required input: {p}')
    return pd.read_parquet(p) if p.suffix.lower()=='.parquet' else pd.read_excel(p) if p.suffix.lower() in {'.xls','.xlsx'} else pd.read_csv(p)
def col(df, candidates, where):
    lc={k.lower():k for k in df.columns}
    for x in candidates:
        if x.lower() in lc:return lc[x.lower()]
    raise KeyError(f'{where}: cannot find any of {candidates}; available={list(df.columns)}')
def key(s):
    # Keep missing IDs missing; astype(str) alone would turn NaN into "nan".
    return s.astype('string').str.strip()

def instance_key_column(df, where):
    """Choose an actual unique instance identifier, never a repeated row index.

    The Combined target files may contain instance_index values that repeat
    across MINLPLib and QPLIB. Do not drop such records or match by position.
    """
    candidates = ['id_global', 'instance_id', 'id', '_key', 'instance_name',
                  'name', 'instance_index']
    available = {c.lower(): c for c in df.columns}
    diagnostic = []
    for candidate in candidates:
        actual = available.get(candidate.lower())
        if actual is None:
            continue
        ids = key(df[actual])
        bad = ids.isna() | ids.eq('')
        duplicates = ids[~bad].duplicated(keep=False)
        diagnostic.append((actual, int(bad.sum()), int(duplicates.sum())))
        if not bad.any() and not duplicates.any():
            print(f'{where}: using unique ID column {actual!r} ({len(df)} rows)')
            return actual
    print(f'{where}: ID diagnostics (column, missing, duplicate rows): {diagnostic}')
    for actual, _, _ in diagnostic:
        print(f'  {actual}: example IDs = {df[actual].head(6).tolist()}')
        ids = key(df[actual]); dup = ids[ids.duplicated(keep=False)]
        if len(dup): print(f'    repeated IDs = {dup.head(8).tolist()}')
    raise ValueError(f'{where}: no unique instance ID column. Inspect the columns above; do not drop duplicates.')
def unique(df,name):
    if df['_key'].isna().any() or df['_key'].duplicated().any():
        raise ValueError(f'{name}: null or duplicate instance keys')
    return df
def gap(ub,lb):return np.maximum(0,np.asarray(ub,dtype=float)-np.asarray(lb,dtype=float))/(np.abs(ub)+np.abs(lb)+ETA)
def canon(s):
    s=str(s).upper().strip()
    if s in {'LP','1','LP_RELAXATION'}:return 'LP'
    if 'SDP' in s or s=='0':return 'SDP_PRIME'
    raise ValueError(f'Unknown relaxation {s}')
def choose(pred):
    if 'selected_relaxation' in pred: return pred['selected_relaxation'].map(canon)
    if 'y_pred' in pred:return pd.Series(np.where(pd.to_numeric(pred['y_pred'])==1,'LP','SDP_PRIME'),index=pred.index)
    if 'y_pred_from_delta' in pred:return pd.Series(np.where(pd.to_numeric(pred['y_pred_from_delta'])==1,'LP','SDP_PRIME'),index=pred.index)
    if 'delta_pred' in pred:return pd.Series(np.where(pd.to_numeric(pred['delta_pred'])>-TAU,'LP','SDP_PRIME'),index=pred.index)
    raise KeyError('Prediction has no selected_relaxation, y_pred or delta_pred')
def discover_run(pattern, specified):
    if specified:return Path(specified)
    hits=sorted(FINAL.glob(pattern))
    if len(hits)!=1:raise RuntimeError(f'Found {len(hits)} directories for {pattern}. Specify its run path explicitly: {hits}')
    return hits[0]
def predictions(run,kind):
    # STATIC Synthetic′ -> Combined ONLY. No dynamic points or dynamic predictions are included.
    paths=[]
    for folder in sorted(run.iterdir()):
        if not folder.is_dir():continue
        s=folder.name.lower()
        if kind=='bb':
            if 'static' not in s or 'classification' not in s and 'regression' not in s:continue
            if not any(x in s for x in ['synthetic','synprime','by_source']):continue
        else:
            if not any(x in s for x in ['synthetic','synprime']):continue
            if 'classification' not in s and 'regression' not in s:continue
        paths.extend(folder.glob('predictions_*.csv' if kind=='bb' else 'preds_*.csv'))
    if not paths:raise FileNotFoundError(f'No Synthetic′ -> Combined static predictions found in {run}')
    models={}
    for p in paths:
        task='classification' if 'classification' in p.parent.name.lower() else 'regression'
        model=p.stem.replace('predictions_','').replace('preds_','').lower()
        if model in {'xgb','gb','rf','xgboost','gradientboosting','randomforest'}:
            label={'xgb':'XGB','gb':'GB','rf':'RF','xgboost':'XGB','gradientboosting':'GB','randomforest':'RF'}[model]+'-'+('C' if task=='classification' else 'R')
        else:label=model.upper()
        if label in models:raise RuntimeError(f'Duplicate prediction model {label}: {p} / {models[label]}')
        df=load(p);k=col(df,['_key','id','id_global'],'prediction ID')
        f=unique(pd.DataFrame({'_key':key(df[k]).map(normalize_id),'selected':choose(df)}),str(p))
        models[label]=f
        print('PRED',kind,label,len(f),p)
    return models


# ==== STATIC GNN saved predictions (LP vs SDP-prime only, to match B&B) ====
def gnn_predictions(run, kind):
    if not Path(run).is_dir():
        raise FileNotFoundError(f'GNN run directory does not exist: {run}')
    files = sorted(Path(run).rglob('preds_test.csv'))
    selected = {}
    for p in files:
        folder=p.parent.name.lower()
        # Root GNN has eight designs. Only LP_vs_SDP_PRIME matches the B&B comparison.
        if kind == 'root':
            if not ('run_07' in folder or 'run_08' in folder):
                continue
        else:
            if not ('synthetic' in folder or 'synprime' in folder or 'run_03' in folder or 'run_04' in folder):
                continue
        task=('classification' if 'classification' in folder or 'run_07' in folder and kind=='root' or 'run_03' in folder and kind=='bb'
              else 'regression' if 'regression' in folder or 'run_08' in folder and kind=='root' or 'run_04' in folder and kind=='bb' else None)
        if task is None:continue
        label='GNN-'+('C' if task=='classification' else 'R')
        if label in selected:
            raise RuntimeError(f'Multiple {label} prediction CSVs in {run}. Select final run. First {selected[label][1]}, second {p}')
        df=load(p)
        k=col(df,['_key','id','id_global'],'GNN prediction ID')
        val=unique(pd.DataFrame({'_key':key(df[k]).map(normalize_id),'selected':choose(df)}),str(p))
        selected[label]=(val,p)
    if set(selected)!= {'GNN-C','GNN-R'}:
        raise RuntimeError(f'{kind} GNN missing saved per-instance preds_test.csv for {set({"GNN-C","GNN-R"})-set(selected)} under {run}. Aggregate result log cannot replace these CSVs.')
    for label,(data,path) in selected.items():
        print('GNN PRED',kind,label,len(data),path)
    return {label:data for label,(data,path) in selected.items()}

def checked_merge_models(handcrafted,gnn):
    if set(handcrafted)&set(gnn):raise ValueError('Duplicate model names')
    return {**handcrafted,**gnn}

def timings():
    raw=load(COMBINED_FEATURE_TIMES)
    idc=col(raw,['id','id_global','_key'],'feature timing ID')
    tc=col(raw,['feature_extraction_time_sec'],'feature time')
    # Combined feature IDs are MINLPLIB__folder__name or QPLIB__folder__QPLIB_xxx.
    # Convert to same canonical key system as reviewbbpredict; strict failure if unmatched.
    z=raw[[idc,tc]].copy();z.columns=['feature_id','feature_time_s']
    z['feature_time_s']=pd.to_numeric(z['feature_time_s'],errors='raise')
    if (z['feature_time_s']<0).any() or not np.isfinite(z['feature_time_s']).all():raise ValueError('Invalid feature times')
    return z

def normalize_id(raw):
    # Matches the canonical combined keys used by root and B&B runners.
    s=str(raw).strip()
    if s.startswith(('MINLPLIB||','QPLIB||')):return s
    if s.upper().startswith('QPLIB__'):
        m=re.search(r'QPLIB[_-]\d+',s,re.I)
        if m:return 'QPLIB||'+m.group(0).upper().replace('-','_')
    if s.upper().startswith('MINLPLIB__'):
        bits=s.split('__');name=bits[1] if len(bits)>1 else bits[-1]
        return 'MINLPLIB||'+name.removesuffix('.pkl')
    return s

def attach_times(base):
    t=timings();t['_key']=t['feature_id'].map(normalize_id)
    # Some saved IDs have source-specific transformations. Never silently substitute position.
    t=t.drop_duplicates(['_key','feature_id'])
    if t['_key'].duplicated().any():
        raise ValueError('Timing key collision: inspect original feature IDs; cannot choose one arbitrarily')
    out=base.merge(t[['_key','feature_time_s']],on='_key',how='left',validate='one_to_one')
    if out['feature_time_s'].isna().any():
        bad=out.loc[out['feature_time_s'].isna(),'_key'].head(15).tolist()
        raise RuntimeError(f'Feature timing unmatched for {out.feature_time_s.isna().sum()} instances. Examples: {bad}. Check canonical IDs.')
    return out

# ==== Bound/time extraction: exact Analysis 2 cumulative solve times ====
def resolve_live_ids(raw_ids):
    """Match LIVE IDs to source-qualified IDs using the existing feature-time index.

    LIVE may use QPLIB_0018 or camshape100 whereas targets and predictions use
    QPLIB||QPLIB_0018 and MINLPLIB||camshape100. Match only unique aliases.
    """
    reference = timings()['feature_id'].map(normalize_id).astype(str)
    if reference.duplicated().any():
        raise ValueError('Nonunique canonical feature-timing reference IDs')
    aliases = {}
    for canonical in reference:
        variants = {canonical, canonical.split('||')[-1]}
        if canonical.startswith('QPLIB||'):
            variants.add(canonical.split('||')[-1].replace('QPLIB_', 'QPLIB-'))
        for alias in variants:
            aliases.setdefault(alias.upper(), set()).add(canonical)
    result=[]; unmatched=set(); ambiguous={}
    for raw in raw_ids:
        value=str(raw).strip()
        found=aliases.get(value.upper(),set()) | aliases.get(normalize_id(value).upper(),set())
        if len(found)==1: result.append(next(iter(found)))
        elif not found: unmatched.add(value); result.append(None)
        else: ambiguous[value]=sorted(found);result.append(None)
    if unmatched or ambiguous:
        raise RuntimeError('Cannot uniquely match LIVE IDs to target IDs: '
            f'{len(unmatched)} unmatched (examples {sorted(unmatched)[:12]}), '
            f'{len(ambiguous)} ambiguous (examples {list(ambiguous.items())[:5]}).')
    print('LIVE ID resolution:',len(set(result)),'unique instances')
    return pd.Series(result,index=raw_ids.index,dtype='string')

def live_lookup():
    p=next((x for x in LIVE_CANDIDATES if x.exists()),None)
    if p is None:raise FileNotFoundError(f'Combined LIVE file missing: {LIVE_CANDIDATES}')
    d=load(p)
    k=col(d,['id_global','instance_id','id','_key','instance_name','name','instance_index'],'LIVE ID')
    r=col(d,['relaxation','relax','method','tree','relaxation_type'],'LIVE relaxation')
    t=col(d,['solve_time','node_solve_time','relax_solve_time','runtime','time','solve_seconds'],'LIVE time')
    u=col(d,['shared_upper_bound','UB','ub','upper_bound','best_ub'],'LIVE UB')
    order=next((c for c in ['node_order','node_index','node_id','node_number','iteration','processed_node'] if c in d),None)
    d=d.copy();d['_key']=resolve_live_ids(key(d[k]));d['_relax']=d[r].map(canon)
    d['_time']=pd.to_numeric(d[t],errors='coerce')
    d['_ub']=pd.to_numeric(d[u],errors='coerce')
    d['_order']=pd.to_numeric(d[order],errors='coerce') if order else np.arange(len(d))
    d['_pos']=np.arange(len(d))
    d=d.sort_values(['_key','_relax','_order','_pos'])
    cumulative={};first={}
    for (name,relax),g in d.groupby(['_key','_relax']):
        ar=g['_time'].to_numpy(float)
        cumulative[(name,relax)]=np.cumsum(ar)
        first[(name,relax)]=ar[0]
    ub=d[np.isfinite(d['_ub'])].groupby('_key')['_ub'].first().to_dict()
    print('LIVE',p,'instances',len(ub))
    print('LIVE relaxation counts:',d['_relax'].value_counts(dropna=False).to_dict())
    print('LIVE sample IDs:',d['_key'].head(7).tolist())
    print('LIVE sample finite upper bounds:',int(np.isfinite(d['_ub']).sum()),'of',len(d))
    print('LIVE sample finite times:',int(np.isfinite(d['_time']).sum()),'of',len(d))
    return ub,cumulative,first

def base_bb(ub,cumulative):
    d=load(TARGET);k=instance_key_column(d,'B&B targets')
    lp=col(d,['lp_obj'],'LP target bound');sdp=col(d,['sdp_prime_obj'],'SDP-prime target bound')
    ln=col(d,['lp_nodes','lp_node_budget','lp_budget_nodes'],'LP node budget')
    sn=col(d,['sdp_nodes','sdp_prime_nodes','sdp_node_budget'],'SDP node budget')
    b=pd.DataFrame({'_key':key(d[k]).map(normalize_id),'lp_lb':pd.to_numeric(d[lp]),'sdp_lb':pd.to_numeric(d[sdp]),
                    'lp_nodes':pd.to_numeric(d[ln]),'sdp_nodes':pd.to_numeric(d[sn])})
    b=unique(b,'B&B targets');b['ub']=b['_key'].map(ub)
    def get(k,r,n):
        a=cumulative.get((k,r));return a[min(int(n),len(a))-1] if a is not None and np.isfinite(n) and int(n)>0 else np.nan
    b['lp_time']= [get(k,'LP',nn) for k,nn in zip(b['_key'],b.lp_nodes)]
    b['sdp_time']=[get(k,'SDP_PRIME',nn) for k,nn in zip(b['_key'],b.sdp_nodes)]
    return b

def resolve_root_ids(raw_names):
    """Resolve plain root-summary names to source-qualified timing IDs.

    Do not infer a library for ambiguous names; all matches must be unique.
    """
    time_df = timings()
    canon_times = time_df['feature_id'].map(normalize_id).astype(str)
    if canon_times.duplicated().any():
        dup = canon_times[canon_times.duplicated(keep=False)].head(10).tolist()
        raise ValueError(f'Nonunique normalized timing IDs: {dup}')

    # Aliases refer to an entire identifier, or to its final instance name.
    aliases = {}
    for raw, canonical in zip(time_df['feature_id'].astype(str), canon_times):
        pieces = raw.split('__')
        alternatives = {raw, canonical, canonical.split('||')[-1]}
        if len(pieces) >= 3:
            # generated id is SOURCE__file_stem__instance_id; both file name
            # and recorded instance_id can be useful, but collisions fail.
            alternatives.update({pieces[1], pieces[-1]})
        for alt in alternatives:
            aliases.setdefault(alt.strip().lower(), set()).add(canonical)

    out=[];missing=[];ambiguous=[]
    for value in raw_names:
        if value is None or pd.isna(value) or not str(value).strip():
            missing.append(str(value));out.append(pd.NA);continue
        orig=str(value).strip()
        v=normalize_id(orig)
        matches=aliases.get(v.lower(), set()) | aliases.get(orig.lower(),set())
        if len(matches)==1:
            out.append(next(iter(matches)))
        elif len(matches)==0:
            missing.append(orig);out.append(pd.NA)
        else:
            ambiguous.append((orig,sorted(matches)[:8]));out.append(pd.NA)
    if missing or ambiguous:
        raise RuntimeError('Root-summary to feature-time ID matching failed: '
                           f'{len(missing)} missing, {len(ambiguous)} ambiguous. '
                           f'Missing examples: {missing[:12]}; ambiguous examples: {ambiguous[:8]}. '
                           'Use explicit source-qualified instance IDs in the source files if ambiguous.')
    print(f'Root summary: resolved {len(out)} instance IDs against feature timing CSV')
    return pd.Series(out,index=raw_names.index,dtype='string')

def base_root(ub,first):
    d=load(SUMMARY)
    k=instance_key_column(d,'root summary')
    l=col(d,['lp_root_obj'],'root LP bound')
    s=col(d,['sdp_root_obj'],'root SDP-prime bound')
    # Root summary has plain names (e.g., camshape100), unlike the
    # source-qualified feature timing, LIVE, and prediction files.
    # Resolve against existing feature time keys, never row order.
    raw_root_keys = key(d[k])
    resolved_root_keys = resolve_root_ids(raw_root_keys)
    b=pd.DataFrame({'_key':resolved_root_keys,'lp_lb':pd.to_numeric(d[l],errors='coerce'),
                    'sdp_lb':pd.to_numeric(d[s],errors='coerce')})
    b=unique(b,'root summary');b['ub']=b['_key'].map(ub)
    b['lp_time']=b['_key'].map(lambda x:first.get((x,'LP'),np.nan))
    b['sdp_time']=b['_key'].map(lambda x:first.get((x,'SDP_PRIME'),np.nan))
    return b

def evaluate(kind,base,models):
    # All methods and baselines share the same test population.
    common=set(base['_key'])
    for df in models.values():common &= set(df['_key'])
    b=base[base['_key'].isin(common)].copy()
    required=['lp_lb','sdp_lb','lp_time','sdp_time','ub','feature_time_s']
    print(f'[{kind}] rows in base={len(base)}, common prediction IDs={len(common)}, matched rows={len(b)}')
    print(f'[{kind}] ID examples (base):',base['_key'].head(5).tolist())
    for label, df in models.items():
        print(f'[{kind}] {label}: {len(df)} prediction IDs; overlap with base={len(set(base["_key"]) & set(df["_key"]))}')
    for c in required:
        z=pd.to_numeric(b[c],errors='coerce')
        print(f'[{kind}] {c}: finite={int(np.isfinite(z).sum())}/{len(b)}; examples=',b[["_key",c]].head(3).to_dict('records'))
    good=np.isfinite(b[required]).all(axis=1)
    if len(b)==0 or not good.any():
        print(f'[{kind}] No evaluable rows. Missingness pattern by column:')
        print(pd.DataFrame({c:~np.isfinite(pd.to_numeric(b[c],errors='coerce')) for c in required}).value_counts(dropna=False).head(15).to_string() if len(b) else '(prediction ID intersection is empty)')
        # Write machine-readable diagnostics before stopping. No data are silently discarded.
        diagnostic=b[['_key']+required].copy()
        for cc in required:
            diagnostic['missing_'+cc]=~np.isfinite(pd.to_numeric(diagnostic[cc],errors='coerce'))
        diag_path=OUT/f'{kind}_missingness_diagnostics.csv'
        diagnostic.to_csv(diag_path,index=False)
        print('\n*** SUMMARY OF DATA AVAILABILITY ***')
        print('base rows=',len(base),'rows after prediction-ID intersection=',len(b))
        for cc in required:
            vals=pd.to_numeric(base[cc],errors='coerce')
            print(f'{cc}: {np.isfinite(vals).sum()} / {len(base)} finite in base; '
                  f'{np.isfinite(pd.to_numeric(b[cc],errors="coerce")).sum()} / {len(b)} after intersection')
        print('Saved diagnostic file:',diag_path)
        print('LIVE keys overlapping base:',len(set(base['_key']) & set(UB)))
        print('LIVE examples:',list(UB)[:6])
        print('Target examples:',base['_key'].head(6).tolist())
        raise RuntimeError(f'{kind}: missing inputs, see {diag_path} and printed column counts. '
                           'Do not assume absence of finite rows means all benchmark instances are invalid.')
    b=b.loc[good].sort_values('_key').reset_index(drop=True)
    print(kind,'common finite population',len(b),'from',len(base),'base instances')
    lp=b.lp_lb.to_numpy(float);sp=b.sdp_lb.to_numpy(float)
    tl=b.lp_time.to_numpy(float);ts=b.sdp_time.to_numpy(float)
    u=b.ub.to_numpy(float);ft=b.feature_time_s.to_numpy(float)
    best_lp=lp>=sp
    choices={'Always LP':np.ones(len(b),dtype=bool),
             "Always SDP'":np.zeros(len(b),dtype=bool),
             'Ideal':best_lp,'Brute Force':best_lp}
    for label,df in models.items():
        merged=b[['_key']].merge(df,on='_key',validate='one_to_one')
        choices[label]=merged['selected'].to_numpy()=='LP'
    points=[];detail=[]
    for label,ch in choices.items():
        lb=np.where(ch,lp,sp);solve=np.where(ch,tl,ts)
        if label=='Brute Force':solve=tl+ts
        overhead=ft if label in models and not label.upper().startswith('GNN') else np.zeros(len(b))
        elapsed=solve+overhead
        z=gap(u,lb)
        points.append({'Experiment':kind,'Strategy':label,'N':len(b),
                       'Avg_Normalized_Value':np.mean(z),'Avg_Time_s':np.mean(elapsed),
                       'Avg_Solver_Time_s':np.mean(solve),'Avg_Feature_Time_s':np.mean(overhead),
                       'Selection_Accuracy_vs_Ideal':np.mean(ch==best_lp)})
        detail.append(pd.DataFrame({'Experiment':kind,'Strategy':label,'_key':b['_key'],
                                    'normalized_gap':z,'selected':np.where(ch,'LP',"SDP'"),
                                    'lp_lb':lp,'sdp_prime_lb':sp,'ub':u,
                                    'solver_time_s':solve,'feature_time_s':overhead,'strategy_time_s':elapsed}))
    p=pd.DataFrame(points);r=pd.concat(detail,ignore_index=True)
    p.to_csv(OUT/f'{kind}_gap_time_points.csv',index=False)
    r.to_csv(OUT/f'{kind}_gap_time_per_instance.csv',index=False)
    return p

# ==== Execute two distinct LP-versus-SDP-prime comparisons (root and B&B) ====
OUT.mkdir(parents=True,exist_ok=True)
actual_root_run=discover_run('sklearn_root_synprime_8016_*',ROOT_RUN)
print('BB_RUN',BB_RUN,'ROOT_RUN',actual_root_run)
if not BB_RUN.is_dir():raise FileNotFoundError(BB_RUN)
bb_models=predictions(BB_RUN,'bb')
root_models=predictions(actual_root_run,'root')
# GNN runs require per-instance saved decisions; never substitute aggregate accuracies.
actual_bb_gnn_run=discover_run('gnn_synprime_8016_*',BB_GNN_RUN)
root_models=checked_merge_models(root_models,gnn_predictions(ROOT_GNN_RUN,'root'))
bb_models=checked_merge_models(bb_models,gnn_predictions(actual_bb_gnn_run,'bb'))
# NOTE: GNN times currently include selected solver only; DI extraction applies
# solely to DI models. Graph preparation and inference require separately recorded
# per-instance GNN times before claiming a full wall-clock time comparison.

UB,CUM,FIRST=live_lookup()
print('LIVE lookup total unique IDs:',len(UB))
bb=attach_times(base_bb(UB,CUM))
print('B&B / LIVE ID intersection:',len(set(bb['_key']) & set(UB)))
root=attach_times(base_root(UB,FIRST))
bb_result=evaluate('bb',bb,bb_models)
root_result=evaluate('root',root,root_models)
all_points=pd.concat([root_result,bb_result],ignore_index=True)
all_points.to_csv(OUT/'analysis3_root_and_bb_gap_time_points.csv',index=False)
print('\nRESULTS\n',all_points.to_string(index=False))
print('\nFiles saved to',OUT)


BB_RUN C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\sklearn_all_runs_synthetic_prime_8016_20261021_014902_034048 ROOT_RUN C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\sklearn_root_synprime_8016_20261021_020512_941535
PRED bb GBC 280 C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\sklearn_all_runs_synthetic_prime_8016_20261021_014902_034048\run_03__synthetic_to_combined__static__classification\predictions_GBC.csv
PRED bb RFC 280 C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\sklearn_all_runs_synthetic_prime_8016_20261021_014902_034048\run_03__synthetic_to_combined__static__classification\predictions_RFC.csv
PRED bb XGB-C 280 C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\sklearn_all_runs_synthetic_prime_8016_20261021_014902_034048\run_03__synthetic_to_combined__static__classification\predictions_XGB-C.csv
PRED bb GBR 280 C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\sklearn_al

In [11]:
# ============================================================
# ANALYSIS 3 — EXACT MODEL FILES (STATIC ONLY, WITH GNN)
# This block replaces the old run-discovery / model-spec part.
# ============================================================

from pathlib import Path
import pandas as pd

FINAL = Path(r"C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib")

# ------------------------------------------------------------
# EXACT RUN DIRECTORIES
# ------------------------------------------------------------

# sklearn root-node transfer run (Synthetic' -> Combined)
ROOT_SK_RUN = FINAL / "sklearn_root_synprime_8016_20261021_020512_941535"

# static root-node GNN run with 8 designs
ROOT_GNN_RUN = FINAL / "gnn_ROOT_8_designs_UNBALANCED_20261024_165255_822932"

# sklearn B&B transfer run (Synthetic' -> Combined)
BB_SK_RUN = FINAL / "sklearn_all_runs_synthetic_prime_8016_20261021_014902_034048"

# static B&B GNN run with 4 designs
BB_GNN_RUN = FINAL / "gnn_synprime_8016_20261021_020453_440418"


# ------------------------------------------------------------
# PANEL DEFINITIONS
# ------------------------------------------------------------
# We need:
#   ROOT:
#     1) LP_vs_SDP         classification
#     2) LP_vs_SDP         regression
#     3) LP_vs_SDP_PRIME   classification
#     4) LP_vs_SDP_PRIME   regression
#
#   B&B:
#     5) LP_vs_SDP_PRIME   classification
#     6) LP_vs_SDP_PRIME   regression
#
# IMPORTANT:
#   - root LP_vs_SDP GNN = run_03 / run_04
#   - root LP_vs_SDP_PRIME GNN = run_07 / run_08
#   - bb   LP_vs_SDP_PRIME GNN = run_03 / run_04
# ------------------------------------------------------------

MODEL_FILES = {
    # ========================================================
    # ROOT NODE — LP vs SDP
    # ========================================================
    ("root", "LP_vs_SDP", "classification"): {
        "XGB-C": ROOT_SK_RUN / "design_03__synthetic_to_combined__classification" / "preds_XGB-C.csv",
        "GBC":   ROOT_SK_RUN / "design_03__synthetic_to_combined__classification" / "preds_GBC.csv",
        "RFC":   ROOT_SK_RUN / "design_03__synthetic_to_combined__classification" / "preds_RFC.csv",
        "GNN-C": ROOT_GNN_RUN / "run_03__LP_vs_SDP__synthetic_to_combined__classification" / "preds_test.csv",
    },

    ("root", "LP_vs_SDP", "regression"): {
        "XGB-R": ROOT_SK_RUN / "design_04__synthetic_to_combined__regression" / "preds_XGB-R.csv",
        "GBR":   ROOT_SK_RUN / "design_04__synthetic_to_combined__regression" / "preds_GBR.csv",
        "RFR":   ROOT_SK_RUN / "design_04__synthetic_to_combined__regression" / "preds_RFR.csv",
        "GNN-R": ROOT_GNN_RUN / "run_04__LP_vs_SDP__synthetic_to_combined__regression" / "preds_test.csv",
    },

    # ========================================================
    # ROOT NODE — LP vs SDP'
    # ========================================================
    ("root", "LP_vs_SDP_PRIME", "classification"): {
        "XGB-C": ROOT_SK_RUN / "design_07__synthetic_to_combined__classification" / "preds_XGB-C.csv",
        "GBC":   ROOT_SK_RUN / "design_07__synthetic_to_combined__classification" / "preds_GBC.csv",
        "RFC":   ROOT_SK_RUN / "design_07__synthetic_to_combined__classification" / "preds_RFC.csv",
        "GNN-C": ROOT_GNN_RUN / "run_07__LP_vs_SDP_PRIME__synthetic_to_combined__classification" / "preds_test.csv",
    },

    ("root", "LP_vs_SDP_PRIME", "regression"): {
        "XGB-R": ROOT_SK_RUN / "design_08__synthetic_to_combined__regression" / "preds_XGB-R.csv",
        "GBR":   ROOT_SK_RUN / "design_08__synthetic_to_combined__regression" / "preds_GBR.csv",
        "RFR":   ROOT_SK_RUN / "design_08__synthetic_to_combined__regression" / "preds_RFR.csv",
        "GNN-R": ROOT_GNN_RUN / "run_08__LP_vs_SDP_PRIME__synthetic_to_combined__regression" / "preds_test.csv",
    },

    # ========================================================
    # SPATIAL B&B — LP vs SDP'
    # ========================================================
    ("bb", "LP_vs_SDP_PRIME", "classification"): {
        "XGB-C": BB_SK_RUN / "run_03__synthetic_to_combined__static__classification" / "predictions_XGB-C.csv",
        "GBC":   BB_SK_RUN / "run_03__synthetic_to_combined__static__classification" / "predictions_GBC.csv",
        "RFC":   BB_SK_RUN / "run_03__synthetic_to_combined__static__classification" / "predictions_RFC.csv",
        "GNN-C": BB_GNN_RUN / "run_03__synthetic_to_combined__classification" / "preds_test.csv",
    },

    ("bb", "LP_vs_SDP_PRIME", "regression"): {
        "XGB-R": BB_SK_RUN / "run_04__synthetic_to_combined__static__regression" / "predictions_XGB-R.csv",
        "GBR":   BB_SK_RUN / "run_04__synthetic_to_combined__static__regression" / "predictions_GBR.csv",
        "RFR":   BB_SK_RUN / "run_04__synthetic_to_combined__static__regression" / "predictions_RFR.csv",
        "GNN-R": BB_GNN_RUN / "run_04__synthetic_to_combined__regression" / "preds_test.csv",
    },
}


# ------------------------------------------------------------
# QUICK FILE CHECK
# ------------------------------------------------------------

missing = []
manifest_rows = []

for (experiment, comparison, task), model_dict in MODEL_FILES.items():
    for model_name, path in model_dict.items():
        exists = path.exists()
        manifest_rows.append({
            "experiment": experiment,
            "comparison": comparison,
            "task": task,
            "model": model_name,
            "path": str(path),
            "exists": exists,
        })
        if not exists:
            missing.append(str(path))

manifest = pd.DataFrame(manifest_rows)
print(manifest.to_string(index=False))

if missing:
    raise FileNotFoundError(
        "Some prediction files are missing.\n\n" + "\n".join(missing)
    )

print("\nALL REQUIRED STATIC MODEL FILES (INCLUDING GNN) WERE FOUND.")


# ------------------------------------------------------------
# OPTIONAL: helper to read all files into one dictionary
# Use this if you want to plug them directly into the rest
# of Analysis 3.
# ------------------------------------------------------------

def load_all_prediction_files(model_files_dict):
    loaded = {}
    for panel, model_dict in model_files_dict.items():
        loaded[panel] = {}
        for model_name, path in model_dict.items():
            loaded[panel][model_name] = pd.read_csv(path)
            print(f"LOADED | {panel} | {model_name} | {len(loaded[panel][model_name])} rows")
    return loaded


ALL_PREDICTIONS = load_all_prediction_files(MODEL_FILES)

print("\nAvailable panels:")
for k in ALL_PREDICTIONS:
    print(" ", k)

experiment      comparison           task model                                                                                                                                                                                                      path  exists
      root       LP_vs_SDP classification XGB-C                       C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\sklearn_root_synprime_8016_20261021_020512_941535\design_03__synthetic_to_combined__classification\preds_XGB-C.csv    True
      root       LP_vs_SDP classification   GBC                         C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\sklearn_root_synprime_8016_20261021_020512_941535\design_03__synthetic_to_combined__classification\preds_GBC.csv    True
      root       LP_vs_SDP classification   RFC                         C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\sklearn_root_synprime_8016_20261021_020512_941535\design_03__synthetic_to_combined__class

FileNotFoundError: Some prediction files are missing.

C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\sklearn_root_synprime_8016_20261021_020512_941535\design_07__synthetic_to_combined__classification\preds_XGB-C.csv
C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\sklearn_root_synprime_8016_20261021_020512_941535\design_07__synthetic_to_combined__classification\preds_GBC.csv
C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\sklearn_root_synprime_8016_20261021_020512_941535\design_07__synthetic_to_combined__classification\preds_RFC.csv
C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\sklearn_root_synprime_8016_20261021_020512_941535\design_08__synthetic_to_combined__regression\preds_XGB-R.csv
C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_qplib\sklearn_root_synprime_8016_20261021_020512_941535\design_08__synthetic_to_combined__regression\preds_GBR.csv
C:\Users\suuser\Desktop\minlplib\Areview\combined_minlplib_

In [13]:
from pathlib import Path

BASE = Path(
    r"C:\Users\suuser\Desktop\minlplib"
    r"\Areview\combined_minlplib_qplib"
)

for run in sorted(BASE.glob("sklearn_root*")):
    if not run.is_dir():
        continue

    print(f"\nRUN: {run.name}")

    for folder in sorted(run.glob("design_*")):
        if folder.is_dir():
            print("  ", folder.name)


RUN: sklearn_root_8_designs_278_two_excluded_20261022_200540_248081

RUN: sklearn_root_8_designs_278_two_excluded_20261022_201010_349221
   design_01__LP_vs_SDP__combined_to_combined__classification
   design_02__LP_vs_SDP__combined_to_combined__regression
   design_03__LP_vs_SDP__synthetic_to_combined__classification
   design_04__LP_vs_SDP__synthetic_to_combined__regression
   design_05__LP_vs_SDP_PRIME__combined_to_combined__classification
   design_06__LP_vs_SDP_PRIME__combined_to_combined__regression
   design_07__LP_vs_SDP_PRIME__synthetic_to_combined__classification
   design_08__LP_vs_SDP_PRIME__synthetic_to_combined__regression

RUN: sklearn_root_8_designs_3_objectives_20261022_185839_521795

RUN: sklearn_root_8_designs_3_objectives_20261022_191035_131945

RUN: sklearn_root_8_designs_3_objectives_20261022_195635_535240

RUN: sklearn_root_8_designs_3_objectives_20261022_195916_270131

RUN: sklearn_root_LP_vs_FULL_SDP_8016_20261022_183421_558909

RUN: sklearn_root_only_same_bb_